# Lifted Learning Graph (LLG) Example

Instantiate a **LiftedLearningGraph** on a minimal Blocks World domain to understand:
- Node feature vectors (`x`)
- Edge labels and indices (`edge_indices`)
- Graph representation for relational planning

The LLG mixes **lifted** (predicates, actions) and **grounded** (facts) nodes.

## 1. Install & Import Packages

In [ ]:
import sys
import os
import numpy as np
import torch
import networkx as nx
from typing import Dict, List, Tuple
from collections import OrderedDict

# Ensure we can import from learner module
sys.path.insert(0, '/home/mariam/goose/learner')
os.chdir('/home/mariam/goose/learner')

print("✓ Core packages imported successfully")

## 2. Define Tiny Domain & Problem

We'll use a minimal **Blocks World** with 2 blocks.

In [ ]:
DOMAIN_PDDL = """
(define (domain blocks)
  (:requirements :typing)
  (:types block - object)
  (:predicates 
    (on ?x ?y - block)
    (clear ?x - block)
    (handempty)
  )
  (:action stack
    :parameters (?x ?y - block)
    :precondition (and (clear ?x) (clear ?y) (handempty))
    :effect (and (on ?x ?y) (not (clear ?y)) (not (handempty)))
  )
)
"""

PROBLEM_PDDL = """
(define (problem blocks-2)
  (:domain blocks)
  (:objects a b - block)
  (:init 
    (clear a)
    (clear b)
    (handempty)
  )
  (:goal (on a b))
)
"""

print("Domain: Blocks World")
print("  Objects: a, b (blocks)")
print("  Predicates: on(x,y), clear(x), handempty()")
print("  Action: stack(?x, ?y)")
print("  Goal: on(a, b)")

## 3. LLG Feature & Edge Definitions

The LLG uses:
- **Node features**: 6 one-hot encoding (P, A, G, N, S, O) + 4 injected position features
- **Edge labels**: neutral, ground, pre_pos, pre_neg, eff_pos, eff_neg

In [ ]:
class LLG_FEATURES:
    """Node feature types (one-hot encoding indices)"""
    P = 0  # is predicate
    A = 1  # is action
    G = 2  # is positive goal (grounded)
    N = 3  # is negative goal (grounded)
    S = 4  # is activated (grounded)
    O = 5  # is object

ENC_FEAT_SIZE = 6  # Number of one-hot features
VAR_FEAT_SIZE = 4  # Injected position feature dimension

LLG_EDGE_LABELS = OrderedDict({
    "neutral": 0,
    "ground": 1,
    "pre_pos": 2,
    "pre_neg": 3,
    "eff_pos": 4,
    "eff_neg": 5,
})

print(f"Node features: {ENC_FEAT_SIZE} one-hot + {VAR_FEAT_SIZE} injected = {ENC_FEAT_SIZE + VAR_FEAT_SIZE} total")
print(f"Edge labels: {list(LLG_EDGE_LABELS.keys())}")

## 4. Instantiate LiftedLearningGraph

Try to import and instantiate the LLG (may encounter torch_geometric issues).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

try:
    from representation import LiftedLearningGraph
    llg = LiftedLearningGraph(DOMAIN_PDDL, PROBLEM_PDDL)
    llg_created = True
    print("✓ LiftedLearningGraph instantiated successfully!")
except Exception as e:
    print(f"⚠ Could not load LLG: {type(e).__name__}")
    print(f"  Reason: {str(e)[:100]}...")
    llg_created = False
    print("\n  Proceeding with manual graph construction for demonstration...")

## 5. Inspect Node Features (x)

In [ ]:
if llg_created:
    print("[Node Features Matrix (x)]")
    print(f"  Shape: {llg.x.shape}")
    print(f"  Dtype: {llg.x.dtype}")
    print(f"  Each row = 1 node feature vector (10-dim)")
    print()
    print("  First 8 nodes:")
    for i in range(min(8, llg.x.shape[0])):
        feat_vals = llg.x[i].numpy()
        one_hot_part = feat_vals[:6]
        injected_part = feat_vals[6:10]
        print(f"    [{i}] one_hot: {one_hot_part.astype(int)}  injected: {injected_part}")
else:
    print("[Manual Construction]")
    print("  Creating example node features manually...")
    # Create example nodes: 2 objects + 3 predicates + goal + action params
    example_nodes = {
        'a': 0,           # object
        'b': 1,           # object
        'on': 2,          # predicate
        'clear': 3,       # predicate
        'handempty': 4,   # predicate
        ('on', ('a', 'b')): 5,  # goal
        'stack': 6,       # action
    }
    
    # Example node features (manually constructed)
    x_manual = torch.zeros(7, 10)
    x_manual[0, LLG_FEATURES.O] = 1  # a is object
    x_manual[1, LLG_FEATURES.O] = 1  # b is object
    x_manual[2, LLG_FEATURES.P] = 1  # on is predicate
    x_manual[3, LLG_FEATURES.P] = 1  # clear is predicate
    x_manual[4, LLG_FEATURES.P] = 1  # handempty is predicate
    x_manual[5, LLG_FEATURES.G] = 1  # goal (on a b)
    x_manual[6, LLG_FEATURES.A] = 1  # action stack
    
    print(f"  Created {x_manual.shape[0]} example nodes")
    print(f"  Shape: {x_manual.shape}")
    print(f"\n  Example node features:")
    for i, (name, idx) in enumerate(example_nodes.items()):
        feat = x_manual[idx]
        print(f"    [{idx}] {str(name):20s}: {feat.numpy()}")

## 6. Inspect Edges & Edge Labels

In [ ]:
if llg_created:
    print("[Edge Indices by Label]")
    from representation.llg import LLG_EDGE_LABELS
    for label_name, label_idx in LLG_EDGE_LABELS.items():
        if label_idx in llg.edge_indices:
            edges = llg.edge_indices[label_idx]
            if edges.numel() > 0:
                n_edges = edges.shape[1]
                print(f"  {label_name:10s} (id={label_idx}): {n_edges:3d} edges")
                # Show first 3 edges
                if n_edges > 0:
                    for j in range(min(3, n_edges)):
                        u, v = edges[0, j].item(), edges[1, j].item()
                        print(f"      ({u} -- {v})")
            else:
                print(f"  {label_name:10s} (id={label_idx}): 0 edges")
else:
    print("[Example Edge List (Manual)]")
    print("  neutral edges (predicate ↔ object):")
    print("    (2 -- 0), (2 -- 1)  # on ↔ {a, b}")
    print("    (3 -- 0), (3 -- 1)  # clear ↔ {a, b}")
    print()
    print("  ground edges (goal structure):")
    print("    (5 -- 2)  # on(a,b) ↔ predicate 'on'")
    print("    (5 -- 0), (5 -- 1)  # goal connects to its arguments")

## 7. Visualize Graph (NetworkX + Matplotlib)

In [ ]:
import matplotlib.pyplot as plt

if llg_created:
    # Use the actual LLG graph
    G = llg.G
    node_labels = {}
    node_colors = {}
    
    # Color scheme: objects=blue, predicates=green, goals=red, actions=orange
    for node in G.nodes():
        if isinstance(node, str):
            if node == 'stack':
                node_colors[node] = 'orange'
                node_labels[node] = 'stack (A)'
            elif node in ['on', 'clear', 'handempty']:
                node_colors[node] = 'green'
                node_labels[node] = node[:3] + ' (P)'
            elif node in ['a', 'b']:
                node_colors[node] = 'blue'
                node_labels[node] = node + ' (O)'
        else:
            if isinstance(node, tuple) and len(node) >= 2:
                node_colors[node] = 'red'
                node_labels[node] = str(node)[:15]
            else:
                node_colors[node] = 'lightgray'
                node_labels[node] = str(node)[:10]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Draw graph
    nx.draw_networkx_nodes(G, pos, 
                          node_color=[node_colors.get(n, 'gray') for n in G.nodes()],
                          node_size=500, ax=ax)
    nx.draw_networkx_labels(G, pos, node_labels, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=1, alpha=0.5, ax=ax)
    
    ax.set_title(f"LiftedLearningGraph: Blocks World\nNodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"Graph visualized: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
else:
    print("Graph construction failed; skipping visualization")

## 8. Unit Tests & Assertions

In [ ]:
if llg_created:
    print("[Running Assertions]")
    
    # Assert 1: Node feature shape
    assert llg.x.shape[1] == ENC_FEAT_SIZE + VAR_FEAT_SIZE, "Node feature size mismatch"
    print(f"  ✓ Node features have correct size: {llg.x.shape[1]}")
    
    # Assert 2: Number of nodes
    assert llg.x.shape[0] == len(llg._node_to_i), "Node count mismatch"
    print(f"  ✓ Node count matches: {llg.x.shape[0]}")
    
    # Assert 3: Edge labels exist
    for label_name, label_idx in LLG_EDGE_LABELS.items():
        assert label_idx in llg.edge_indices, f"Missing edge label: {label_name}"
    print(f"  ✓ All {len(LLG_EDGE_LABELS)} edge labels present")
    
    # Assert 4: Graph structure
    assert llg.num_nodes > 0, "Graph has no nodes"
    assert llg.num_edges >= 0, "Graph has negative edges"
    print(f"  ✓ Graph structure valid: {llg.num_nodes} nodes, {llg.num_edges} edges")
    
    # Assert 5: Feature values are valid
    assert llg.x.min() >= -1.0 and llg.x.max() <= 1.0, "Feature values out of range"
    print(f"  ✓ Feature values in valid range: [{llg.x.min():.3f}, {llg.x.max():.3f}]")
    
    print("\n✓ All assertions passed!")
else:
    print("[Manual Assertions]")
    assert x_manual.shape == (7, 10), "Manual feature matrix shape incorrect"
    assert x_manual.sum(dim=1)[:5].sum() == 5, "One-hot encoding check failed"
    print("  ✓ Manual constructions valid")

## 9. Summary & Key Takeaways

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║           LIFTED LEARNING GRAPH (LLG) - KEY INSIGHTS             ║
╚══════════════════════════════════════════════════════════════════╝

1. NODE TYPES (Features):
   • Predicates (P):      Lifted relations from domain
   • Objects (O):         Domain objects
   • Actions (A):         Action schemas
   • Goals (G/N):         Grounded goal facts (positive/negative)
   • Activated (S):       Ground facts true in current state
   
2. EDGE TYPES (Labels):
   • neutral:   Unspecified relations (e.g., pred ↔ object)
   • ground:    Grounding: fact ↔ predicate, fact ↔ arguments
   • pre_pos:   Positive precondition relations
   • pre_neg:   Negative precondition relations
   • eff_pos:   Positive effect relations
   • eff_neg:   Negative effect relations

3. NODE FEATURES (10-dim vector):
   • Dims 0-5:   One-hot encoding of node type (6 types)
   • Dims 6-9:   Injected position features (4-dim unit vector)
                 → Distinguishes argument positions: on(A,B) vs on(B,A)

4. WHY LIFTED?
   • Mixes lifted (predicates, actions) with grounded (facts)
   • Enables GNN to learn patterns at both abstraction levels
   • Shares representations across different problem instances
   • Scales to new problems with same domain structure

5. GRAPH CONSTRUCTION:
   ① Add domain objects and predicates (fully connected)
   ② Add goal facts and connect to predicates via variable nodes
   ③ Add action schemas with parameters
   ④ Connect preconditions/effects with appropriate edge labels

6. STATE REPRESENTATION:
   • state_to_tensor() extends graph with activated facts
   • Reuses existing nodes for goal facts (S feature ↓ 1)
   • Adds new nodes for other activated ground facts
   • Updates edge_indices accordingly

════════════════════════════════════════════════════════════════════
""")

if llg_created:
    print(f"Actual graph stats from Blocks domain:")
    print(f"  Nodes: {llg.num_nodes}")
    print(f"  Edges: {llg.num_edges}")
    print(f"  Lifted: {llg.lifted}")
    print(f"  Node feature dim: {llg.n_node_features}")
else:
    print(f"Manual example stats:")
    print(f"  Nodes: 7 (2 objects + 3 predicates + 1 goal + 1 action)")
    print(f"  Edges: ~10 (neutral + ground relations)")
    print(f"  Node feature dim: {ENC_FEAT_SIZE + VAR_FEAT_SIZE}")